# Parse MIDI into note sequences (LSTM input)

Goal here is to turn every filtered MIDI file into a sequence of notes (pitch, duration, velocity, offset from the start of the piece) that we can feed into the LSTM later.

Input: `data/raw/<composer>/*.mid` (built by `scripts/filter_dataset.py`)

Output: `data/processed/lstm/<composer>/<file>.npz` + a manifest csv with sequence lengths so we can sanity check stuff

In [1]:
import warnings
import pretty_midi
import numpy as np
import pandas as pd
from pathlib import Path

# pretty_midi likes to complain about tempo/key events on non-zero tracks
# for a lot of these old midi files, doesn't actually break anything
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed/lstm")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(RAW_DIR / "manifest.csv")
manifest.head()

,composer,filename,source_path
0,bach,AveMaria.mid,Bach\AveMaria.mid
1,bach,01_Menuet.mid,Bach\Bwv ''Little Notebook for Anna Magdalena ...
2,bach,02_Menuet.mid,Bach\Bwv ''Little Notebook for Anna Magdalena ...
3,bach,03_Menuet.mid,Bach\Bwv ''Little Notebook for Anna Magdalena ...
4,bach,04_Menuet.mid,Bach\Bwv ''Little Notebook for Anna Magdalena ...


In [3]:
def parse_midi_to_sequence(path):
    """grabs every non-drum note across all instruments and sorts by start time"""
    pm = pretty_midi.PrettyMIDI(str(path))

    notes = []
    for inst in pm.instruments:
        if inst.is_drum:
            continue
        notes.extend(inst.notes)

    if len(notes) == 0:
        raise ValueError("no notes found")

    notes.sort(key=lambda n: n.start)

    pitch = np.array([n.pitch for n in notes], dtype=np.int16)
    duration = np.array([n.end - n.start for n in notes], dtype=np.float32)
    velocity = np.array([n.velocity for n in notes], dtype=np.int16)
    # offset = time since previous note started, first note gets 0
    starts = np.array([n.start for n in notes], dtype=np.float32)
    offset = np.diff(starts, prepend=starts[0])

    return pitch, duration, velocity, offset

In [4]:
# quick test on one file before running the whole thing
test_row = manifest.iloc[0]
test_path = RAW_DIR / test_row["composer"] / test_row["filename"]
p, d, v, o = parse_midi_to_sequence(test_path)
print(test_row["filename"], "-> seq len:", len(p))
print("pitch sample:", p[:10])
print("duration sample:", d[:10])

AveMaria.mid -> seq len: 793
pitch sample: [53 41 57 60 65 69 60 65 69 53]
duration sample: [0.125 2.    0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125]


In [5]:
results = []
failed = []

for i, row in manifest.iterrows():
    composer = row["composer"]
    filename = row["filename"]
    src = RAW_DIR / composer / filename

    try:
        pitch, duration, velocity, offset = parse_midi_to_sequence(src)
    except Exception as e:
        failed.append({"composer": composer, "filename": filename, "error": str(e)})
        continue

    comp_dir = OUT_DIR / composer
    comp_dir.mkdir(exist_ok=True)
    out_path = comp_dir / (Path(filename).stem + ".npz")
    np.savez_compressed(out_path, pitch=pitch, duration=duration, velocity=velocity, offset=offset)

    results.append({
        "composer": composer,
        "filename": filename,
        "npz_path": str(out_path.relative_to(OUT_DIR.parent.parent)),
        "seq_len": len(pitch),
    })

    if i % 300 == 0:
        print(f"...{i}/{len(manifest)} done")

print("finished. ok:", len(results), "failed:", len(failed))

...0/1637 done


...300/1637 done


...600/1637 done


...900/1637 done


...1200/1637 done


...1500/1637 done


finished. ok: 1635 failed: 2


In [6]:
seq_df = pd.DataFrame(results)
seq_df.to_csv("../data/processed/lstm_manifest.csv", index=False)

fail_df = pd.DataFrame(failed)
fail_df.to_csv("../data/processed/lstm_failed.csv", index=False)

seq_df.groupby("composer")["seq_len"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
composer,,,,
bach,1024,1733.605469,83,33004
beethoven,219,7707.082192,90,46897
chopin,136,2350.757353,164,19929
mozart,256,5383.554688,122,17850


In [7]:
fail_df

,composer,filename,error
0,beethoven,Anhang_14-3.mid,Could not decode key with 3 flats and mode 255
1,mozart,K281_Piano_Sonata_n03_3mov.mid,Could not decode key with 2 flats and mode 2


## Notes

- 2 files failed to parse (checked below), just skipped them, not worth debugging weird corrupt midi files for 2 out of 1637
- sequence lengths are all over the place (some pieces have like 80 notes, some have 40k+), so whatever model we build is gonna need padding/truncation, that's a problem for the model building step not this one
- saved everything as compressed npz so we don't reparse midi files every time we touch the LSTM notebook